In [1]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Recupera il token dai Kaggle Secrets
user_secrets = UserSecretsClient()
HUGGINGFACE_TOKEN = user_secrets.get_secret("<YOUR_HUGGINGFACE_TOKEN>")

# 🔐 Login Hugging Face
login(token=HUGGINGFACE_TOKEN)

print("Login successful")

Login successful


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

GEN_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Carica modello base
base_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2. Applica i pesi LoRA dal dataset Kaggle
model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/input/mistral-ppo-trained-model-on-short-jokes"
)

model.eval()

# 3. Carica tokenizer dal dataset (stesso path)
tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/mistral-ppo-trained-model-on-short-jokes",
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token


2025-07-31 10:35:38.746004: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753958138.939160      72 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753958138.997610      72 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [3]:
def make_prompt(user_prompt):
    return f"""<s>[SYSTEM] 
        You are a professional comedy writer who specializes in one-liners and dad jokes. 
        Your style is irreverent, witty, and playful — sometimes exaggerated, but always clever and tailored to the user's prompt. 
        You never explain the joke. You never break character. Just deliver a short, sharp punchline that fits the context.
        Always be original, avoid clichés, and keep it brief — one or two lines at most. 
        If the user simply asks for a joke, invent one on the spot. Your job is to make people laugh.
        If the user asks to explain a joke or asks something that is not relevant in the context of jokes, 
        make fun of them, but don't be offensive.
        \n[USER] {user_prompt} . \n [ASSISTANT]</s>"""

def generate_response(user_input, temperature=1.2, max_new_tokens=128):
    prompt = make_prompt(user_input)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_ids = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)
    
    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            attention_mask = attention_mask,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


In [ ]:
## Alternativa 1
while True:
    user_input = input("Tu: ")
    if user_input.lower() in ["exit", "quit", "q"]:
        break
    response = generate_response(user_input)
    print(f"Modello: {response}\n")


In [5]:
from IPython.display import display
import ipywidgets as widgets

text_input = widgets.Text(
    placeholder='Scrivi un prompt...',
    continuous_update=False  # solo invio esplicito momentaneo
)
submit_button = widgets.Button(description="Invia", button_style="primary")
output = widgets.Output()

def handle_submit(_):
    with output:
        output.clear_output()
        resp = generate_response(text_input.value)
        print(f"Tu: {text_input.value}\n\nModello: {resp}")

# Collega il bottone
submit_button.on_click(handle_submit)

# Mostra i widget
display(text_input, submit_button, output)



Text(value='', continuous_update=False, placeholder='Scrivi un prompt...')

Button(button_style='primary', description='Invia', style=ButtonStyle())

Output()

-----

### Chatbot

In [ ]:
# Parametri per chat con memoria
MAX_TURNS_MEMORY = 6       # Quanti "turni" conservare (utente + modello)
MAX_TOKENS_CONTEXT = 6000  # Soglia sicurezza (su 8k del modello Mistral)


In [ ]:
chat_history = []  # Memorizza i turni della conversazione

def make_chat_prompt(user_message):
    """
    Costruisce un prompt includendo gli ultimi turni della conversazione.
    """
    # Aggiungi nuovo turno utente allo storico
    chat_history.append({"role": "user", "content": user_message})

    # Mantieni solo gli ultimi MAX_TURNS_MEMORY turni
    recent_history = chat_history[-MAX_TURNS_MEMORY:]

    # Costruisci prompt concatenando sistema e cronologia
    prompt_parts = ["<s>[SYSTEM] You are a witty comedy chatbot specializing in one-liners and dad jokes."]
    for turn in recent_history:
        if turn["role"] == "user":
            prompt_parts.append(f"[USER] {turn['content']}")
        else:
            prompt_parts.append(f"[ASSISTANT] {turn['content']}")
    prompt_parts.append("[ASSISTANT]")

    return "\n".join(prompt_parts)


In [ ]:
def generate_response_with_context(user_message, temperature=1.2, max_new_tokens=128):
    """
    Genera risposta usando il contesto conversazionale recente.
    """
    prompt = make_chat_prompt(user_message)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_ids = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)

    # Se il prompt eccede MAX_TOKENS_CONTEXT, tronca dallo storico
    if input_ids.shape[1] > MAX_TOKENS_CONTEXT:
        # Rimuovi turni più vecchi e ricrea prompt
        while input_ids.shape[1] > MAX_TOKENS_CONTEXT and len(chat_history) > 2:
            chat_history.pop(0)  # Rimuove il turno più vecchio
            prompt = make_chat_prompt(user_message)
            inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
            input_ids = inputs["input_ids"].to(DEVICE)
            attention_mask = inputs["attention_mask"].to(DEVICE)

    # Generazione
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            attention_mask=attention_mask,  # <-- Passa attention_mask qui
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        output_ids[0][input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Aggiungi risposta allo storico
    chat_history.append({"role": "assistant", "content": response})
    return response



In [ ]:
# Alternativa 1
print("Chat attiva. Digita 'exit' per terminare.\n")
while True:
    user_input = input("Tu: ")
    if user_input.lower() in ["exit", "quit", "q"]:
        break
    response = generate_response_with_context(user_input)
    print(f"Modello: {response}\n")


In [ ]:
from IPython.display import display
import ipywidgets as widgets

# Campo di input per il messaggio
text_input = widgets.Text(
    placeholder='Scrivi un messaggio...',
    continuous_update=False  # evita trigger non voluti
)

# Bottone di invio
submit_button = widgets.Button(description="Invia", button_style="primary")

# Area di output
output = widgets.Output()

# Funzione callback su click del bottone
def handle_submit(_):
    with output:
        output.clear_output()
        resp = generate_response_with_context(text_input.value)
        print(f"Prompt: {text_input.value}\n\nRisposta: {resp}")

# Collega il bottone alla funzione
submit_button.on_click(handle_submit)

# Mostra i widget
display(text_input, submit_button, output)

